# Performance Tips

## Compilation

jax-healpy functions are JIT-compiled on their first call for a given set of input shapes, dtypes and
static arguments (such as `nside`). Wrapping your own pipelines with `jax.jit` lets XLA fuse the
operations. The resolution parameter `nside` determines the array shapes, so it must be declared
static:

In [ ]:
import time
from functools import partial

import jax
import jax.numpy as jnp

import jax_healpy as hp

jax.config.update('jax_enable_x64', True)


@partial(jax.jit, static_argnames='nside')
def roundtrip(nside, pixels):
    theta, phi = hp.pix2ang(nside, pixels)
    return hp.ang2pix(nside, theta, phi)


nside = 128
pixels = jnp.arange(hp.nside2npix(nside))

# The first call triggers the compilation
roundtrip(nside, pixels).block_until_ready()

# Subsequent calls with the same shapes and nside reuse the compiled function
start = time.perf_counter()
roundtrip(nside, pixels).block_until_ready()
print(f'Round-trip time: {time.perf_counter() - start:.4f} seconds')

JAX dispatches computations asynchronously, so call `block_until_ready()` before stopping a timer.
Each new value of `nside` triggers a new compilation.

The cost of the compilation is paid once per `nside`. The plot below compares, for
several resolutions, the time of the first call, which includes the compilation, with the
time of the following calls. At low resolution, the compilation dominates by orders of
magnitude.

In [ ]:
import matplotlib.pyplot as plt


def timed(nside, pixels):
    start = time.perf_counter()
    roundtrip(nside, pixels).block_until_ready()
    return time.perf_counter() - start


# nside = 128 is left out, since it has already been compiled above
nsides = [16, 32, 64, 256, 512, 1024]
first_call, next_calls = [], []
for nside_ in nsides:
    pixels_ = jnp.arange(hp.nside2npix(nside_))
    first_call.append(timed(nside_, pixels_))
    next_calls.append(min(timed(nside_, pixels_) for _ in range(5)))

fig, ax = plt.subplots(figsize=(6, 4))
npixs = [hp.nside2npix(n) for n in nsides]
ax.plot(npixs, first_call, 'o-', color='#2a78d6', label='First call (with compilation)')
ax.plot(npixs, next_calls, 'o-', color='#eb6834', label='Following calls')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Number of pixels')
ax.set_ylabel('Time of pix2ang + ang2pix [s]')
ax.grid(alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False)
plt.show()

Any argument that sets the shape of an array must be static, not only `nside`: for
example, the number of points drawn at random in this function.

In [ ]:
@partial(jax.jit, static_argnames=['nside', 'n_samples'])
def random_pixels(key, nside, n_samples):
    key_theta, key_phi = jax.random.split(key)
    theta = jnp.arccos(jax.random.uniform(key_theta, (n_samples,), minval=-1, maxval=1))
    phi = jax.random.uniform(key_phi, (n_samples,), maxval=2 * jnp.pi)
    return hp.ang2pix(nside, theta, phi)


print(random_pixels(jax.random.key(0), nside, 5))

## GPU Acceleration

The operations run on the default JAX device, which is the GPU if JAX has been installed with GPU support:

In [ ]:
print(f'JAX backend: {jax.default_backend()}')
print(f'JAX devices: {jax.devices()}')

## Batch Processing

Pass whole arrays to the functions instead of looping over their elements in Python: a
single call processes all the elements in one compiled operation, whereas a loop pays the
overhead of a call for each element.

In [ ]:
n_points = 1000
theta_points = jnp.arccos(jax.random.uniform(jax.random.key(1), (n_points,), minval=-1, maxval=1))
phi_points = jax.random.uniform(jax.random.key(2), (n_points,), maxval=2 * jnp.pi)
# Compile for both shapes: an array of n_points and a scalar
hp.ang2pix(nside, theta_points, phi_points).block_until_ready()
hp.ang2pix(nside, theta_points[0], phi_points[0]).block_until_ready()

start = time.perf_counter()
hp.ang2pix(nside, theta_points, phi_points).block_until_ready()
time_vectorized = time.perf_counter() - start

start = time.perf_counter()
pixels_loop = [hp.ang2pix(nside, t, p) for t, p in zip(theta_points, phi_points)]
jax.block_until_ready(pixels_loop)
time_loop = time.perf_counter() - start

print(f'One call on {n_points} points: {time_vectorized * 1e3:.2f} ms')
print(f'{n_points} calls on one point: {time_loop * 1e3:.0f} ms')

Most functions broadcast over their array arguments. For more complex pipelines, `jax.vmap` vectorizes
a function over a batch dimension:

In [ ]:
# Interpolate a batch of maps at the same locations
batch_size = 10
npix = hp.nside2npix(nside)
batch_maps = jax.random.normal(jax.random.key(0), (batch_size, npix))
theta = jnp.array([0.5, 1.0, 1.5])
phi = jnp.array([0.0, 1.0, 2.0])

values = jax.vmap(hp.get_interp_val, in_axes=(0, None, None))(batch_maps, theta, phi)
print(values.shape)  # (10, 3)

## Memory Management

For very large operations that do not fit in the device memory, process the data in chunks:

In [ ]:
def ang2pix_in_chunks(nside, theta, phi, chunk_size=1_000_000):
    results = []
    for start in range(0, theta.size, chunk_size):
        stop = start + chunk_size
        results.append(hp.ang2pix(nside, theta[start:stop], phi[start:stop]))
    return jnp.concatenate(results)

By default, JAX preallocates 75% of the GPU memory on its first operation. This behavior can be changed
through [environment variables](https://docs.jax.dev/en/latest/gpu_memory_allocation.html), for
example `XLA_PYTHON_CLIENT_PREALLOCATE=false`.

Using 32-bit precision halves the memory footprint, but the results then diverge from healpy even at
moderate resolution, so 64-bit precision is recommended (see [Getting Started](getting-started.md)).